# SpecMambaNet: Medical Image Segmentation
**Full Pipeline: Preprocessing → Training → Evaluation**

Supports:
- **ACDC** (Cardiac MRI, 4 classes: BG/RV/MYO/LV)
- **ImageCAS** (Coronary CTA, binary: BG/Vessel)

Architecture: Dual-Stream FFT + PseudoMamba in U-Net Encoder-Decoder + Deep Supervision

## 0. Setup & Installation

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['nibabel', 'albumentations', 'einops']:
    try:
        __import__(pkg)
    except ImportError:
        install(pkg)

import os, glob, json, math, configparser
from datetime import datetime
from collections import defaultdict, OrderedDict
from typing import Optional

import numpy as np
import nibabel as nib
from skimage.transform import resize
from scipy.ndimage import distance_transform_edt, binary_erosion
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

try:
    import albumentations as A
    HAS_ALB = True
except ImportError:
    HAS_ALB = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration

In [ ]:
# ====== CHOOSE DATASET ======
DATASET = 'acdc'  # 'acdc' or 'imagecas'

# Paths (adjust for Kaggle)
CFG = {
    'acdc': {
        'raw_dir': '/kaggle/input/acdc-dataset',  # NIfTI files
        'prep_dir': '/kaggle/working/preprocessed/ACDC',
        'num_classes': 4,
        'class_names': ['BG', 'RV', 'MYO', 'LV'],
        'in_channels': 3,
        'img_size': 224,
    },
    'imagecas': {
        'raw_dir': '/kaggle/input/imagecas',  # img.nii.gz + label.nii.gz
        'prep_dir': '/kaggle/working/preprocessed/ImageCAS',
        'num_classes': 2,
        'class_names': ['BG', 'Vessel'],
        'in_channels': 3,
        'img_size': 224,
    }
}

# Training hyperparams
TRAIN_CFG = {
    'batch_size': 4,
    'epochs': 200,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'warmup_epochs': 10,
    'early_stop': 40,
    'base_channels': 48,
    'use_amp': True,
    'augment': True,
    'deep_supervision': True,
}

cfg = CFG[DATASET]
print(f'Dataset: {DATASET.upper()} | Classes: {cfg["num_classes"]} | Size: {cfg["img_size"]}')

## 2. Preprocessing

In [ ]:
def normalize_zscore(image):
    p05, p995 = np.percentile(image, 0.5), np.percentile(image, 99.5)
    image = np.clip(image, p05, p995)
    mean, std = np.mean(image), np.std(image)
    return (image - mean) / std if std > 0 else image - mean


def preprocess_acdc(raw_dir, output_dir, target_size=224):
    """Preprocess ACDC: read NIfTI ED/ES frames, save as .npy with spacing."""
    ts = (target_size, target_size)
    for split in ['training', 'testing']:
        split_input = os.path.join(raw_dir, split)
        if not os.path.isdir(split_input):
            print(f'  Skip {split} (not found)')
            continue
        
        split_out = os.path.join(output_dir, split)
        vol_dir = os.path.join(split_out, 'volumes')
        msk_dir = os.path.join(split_out, 'masks')
        os.makedirs(vol_dir, exist_ok=True)
        os.makedirs(msk_dir, exist_ok=True)
        
        patients = sorted([d for d in os.listdir(split_input)
                           if os.path.isdir(os.path.join(split_input, d)) and d.startswith('patient')])
        
        volume_info = {}
        for pat in tqdm(patients, desc=f'ACDC {split}'):
            pat_path = os.path.join(split_input, pat)
            info_path = os.path.join(pat_path, 'Info.cfg')
            if not os.path.exists(info_path):
                continue
            
            parser = configparser.ConfigParser()
            with open(info_path) as f:
                parser.read_string('[DEFAULT]\n' + f.read())
            ed = int(parser['DEFAULT']['ED'])
            es = int(parser['DEFAULT']['ES'])
            
            for frame_num, tag in [(ed, 'ED'), (es, 'ES')]:
                img_path = os.path.join(pat_path, f'{pat}_frame{frame_num:02d}.nii.gz')
                msk_path = os.path.join(pat_path, f'{pat}_frame{frame_num:02d}_gt.nii.gz')
                if not os.path.exists(img_path):
                    img_path = img_path.replace('.nii.gz', '.nii')
                    msk_path = msk_path.replace('.nii.gz', '.nii')
                if not os.path.exists(img_path) or not os.path.exists(msk_path):
                    continue
                
                nii = nib.load(img_path)
                img = normalize_zscore(nii.get_fdata())
                msk = nib.load(msk_path).get_fdata()
                orig_shape = img.shape
                orig_sp = nii.header.get_zooms()
                D = img.shape[2]
                
                eff_sp = [
                    float(orig_sp[2]),
                    float(orig_sp[0]) * orig_shape[0] / ts[0],
                    float(orig_sp[1]) * orig_shape[1] / ts[1],
                ]
                
                vol = np.zeros((*ts, D), dtype=np.float32)
                seg = np.zeros((*ts, D), dtype=np.uint8)
                for i in range(D):
                    vol[:,:,i] = resize(img[:,:,i], ts, order=1, preserve_range=True, anti_aliasing=True)
                    seg[:,:,i] = resize(msk[:,:,i], ts, order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)
                
                vid = f'{pat}_{tag}'
                np.save(os.path.join(vol_dir, f'{vid}.npy'), vol)
                np.save(os.path.join(msk_dir, f'{vid}.npy'), seg)
                volume_info[vid] = {
                    'num_slices': D,
                    'orig_shape': [int(s) for s in orig_shape],
                    'orig_spacing': [float(s) for s in orig_sp],
                    'effective_spacing': eff_sp,
                }
        
        meta = {'dataset': 'ACDC', 'num_classes': 4,
                'class_names': ['BG', 'RV', 'MYO', 'LV'],
                'target_size': list(ts),
                'total_volumes': len(volume_info), 'volume_info': volume_info}
        with open(os.path.join(split_out, 'metadata.json'), 'w') as f:
            json.dump(meta, f, indent=2)
        print(f'  {split}: {len(volume_info)} volumes')


def preprocess_imagecas(raw_dir, output_dir, target_size=224):
    """Preprocess ImageCAS: CTA volumes → 2D slices as .npy with spacing.
    
    Expected structure:
      raw_dir/
        ├── 0001/img.nii.gz, label.nii.gz
        ├── 0002/img.nii.gz, label.nii.gz
        └── ... (or flat: 0001_img.nii.gz, 0001_label.nii.gz)
    """
    ts = (target_size, target_size)
    
    # Detect structure
    cases = []
    for item in sorted(os.listdir(raw_dir)):
        item_path = os.path.join(raw_dir, item)
        if os.path.isdir(item_path):
            img_p = os.path.join(item_path, 'img.nii.gz')
            lbl_p = os.path.join(item_path, 'label.nii.gz')
            if not os.path.exists(img_p):
                img_p = os.path.join(item_path, 'img.nii')
                lbl_p = os.path.join(item_path, 'label.nii')
            if os.path.exists(img_p) and os.path.exists(lbl_p):
                cases.append((item, img_p, lbl_p))
        elif item.endswith(('_img.nii.gz', '_img.nii')):
            cid = item.split('_img')[0]
            lbl_p = os.path.join(raw_dir, item.replace('_img', '_label'))
            if os.path.exists(lbl_p):
                cases.append((cid, item_path, lbl_p))
    
    if not cases:
        print('No ImageCAS cases found. Check raw_dir structure.')
        return
    
    n_train = int(len(cases) * 0.8)
    splits = [('training', cases[:n_train]), ('testing', cases[n_train:])]
    
    for split_name, split_cases in splits:
        split_out = os.path.join(output_dir, split_name)
        vol_dir = os.path.join(split_out, 'volumes')
        msk_dir = os.path.join(split_out, 'masks')
        os.makedirs(vol_dir, exist_ok=True)
        os.makedirs(msk_dir, exist_ok=True)
        
        volume_info = {}
        for cid, img_path, lbl_path in tqdm(split_cases, desc=f'ImageCAS {split_name}'):
            try:
                nii = nib.load(img_path)
                img = normalize_zscore(nii.get_fdata().astype(np.float32))
                lbl = (nib.load(lbl_path).get_fdata() > 0).astype(np.uint8)
                orig_shape = img.shape
                orig_sp = nii.header.get_zooms()
                
                # ImageCAS: subsample slices with vessels + random BG slices
                vessel_slices = [i for i in range(img.shape[2]) if lbl[:,:,i].sum() > 10]
                bg_slices = [i for i in range(img.shape[2]) if i not in vessel_slices]
                n_bg = min(len(bg_slices), len(vessel_slices) // 2)
                np.random.seed(42)
                selected = sorted(vessel_slices + list(np.random.choice(bg_slices, n_bg, replace=False))) if n_bg > 0 else vessel_slices
                
                if len(selected) == 0:
                    continue
                
                D = len(selected)
                vol = np.zeros((*ts, D), dtype=np.float32)
                seg = np.zeros((*ts, D), dtype=np.uint8)
                for j, si in enumerate(selected):
                    vol[:,:,j] = resize(img[:,:,si], ts, order=1, preserve_range=True, anti_aliasing=True)
                    seg[:,:,j] = resize(lbl[:,:,si], ts, order=0, preserve_range=True, anti_aliasing=False).astype(np.uint8)
                
                eff_sp = [
                    float(orig_sp[2]),
                    float(orig_sp[0]) * orig_shape[0] / ts[0],
                    float(orig_sp[1]) * orig_shape[1] / ts[1],
                ]
                
                vid = f'case_{cid}'
                np.save(os.path.join(vol_dir, f'{vid}.npy'), vol)
                np.save(os.path.join(msk_dir, f'{vid}.npy'), seg)
                volume_info[vid] = {
                    'num_slices': D,
                    'orig_shape': [int(s) for s in orig_shape],
                    'orig_spacing': [float(s) for s in orig_sp],
                    'effective_spacing': eff_sp,
                    'selected_slices': selected,
                }
            except Exception as e:
                print(f'  Error {cid}: {e}')
        
        meta = {'dataset': 'ImageCAS', 'num_classes': 2,
                'class_names': ['BG', 'Vessel'],
                'target_size': list(ts),
                'total_volumes': len(volume_info), 'volume_info': volume_info}
        with open(os.path.join(split_out, 'metadata.json'), 'w') as f:
            json.dump(meta, f, indent=2)
        print(f'  {split_name}: {len(volume_info)} volumes')

In [ ]:
# Run preprocessing
prep_dir = cfg['prep_dir']
train_dir = os.path.join(prep_dir, 'training')

if not os.path.exists(os.path.join(train_dir, 'metadata.json')):
    print(f'Preprocessing {DATASET.upper()}...')
    if DATASET == 'acdc':
        preprocess_acdc(cfg['raw_dir'], prep_dir, cfg['img_size'])
    else:
        preprocess_imagecas(cfg['raw_dir'], prep_dir, cfg['img_size'])
else:
    print('Preprocessed data found, skipping.')

with open(os.path.join(train_dir, 'metadata.json')) as f:
    meta = json.load(f)
print(f'Volumes: {meta["total_volumes"]} | Classes: {meta["num_classes"]}')
sample = list(meta['volume_info'].values())[0]
print(f'Sample spacing (mm): {[round(s,2) for s in sample["effective_spacing"]]}')

## 3. Dataset & Augmentation

In [ ]:
def get_augmentations():
    if not HAS_ALB:
        return None
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5, border_mode=0),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.1, 0.1), p=0.5),
        A.ElasticTransform(alpha=50, sigma=5, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    ])


class SliceDataset(Dataset):
    """Universal 2D slice dataset for any preprocessed medical volume."""
    
    def __init__(self, npy_dir, in_channels=3, augment=False):
        self.in_channels = in_channels
        self.transform = get_augmentations() if augment else None
        self._cache = OrderedDict()
        self.max_cache = 20
        
        self.vol_paths = sorted(glob.glob(os.path.join(npy_dir, 'volumes', '*.npy')))
        self.msk_paths = sorted(glob.glob(os.path.join(npy_dir, 'masks', '*.npy')))
        
        meta_path = os.path.join(npy_dir, 'metadata.json')
        vol_info = {}
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                vol_info = json.load(f).get('volume_info', {})
        
        self.index_map = []
        for i, vp in enumerate(self.vol_paths):
            vid = os.path.basename(vp).replace('.npy', '')
            n = vol_info[vid]['num_slices'] if vid in vol_info else np.load(vp, mmap_mode='r').shape[2]
            for s in range(n):
                self.index_map.append((i, s))
        
        print(f'SliceDataset: {len(self.index_map)} slices from {len(self.vol_paths)} volumes (aug={augment})')
    
    def _load(self, idx):
        if idx in self._cache:
            self._cache.move_to_end(idx)
            return self._cache[idx]
        v = np.load(self.vol_paths[idx], mmap_mode='r')
        m = np.load(self.msk_paths[idx], mmap_mode='r')
        self._cache[idx] = (v, m)
        if len(self._cache) > self.max_cache:
            self._cache.popitem(last=False)
        return v, m
    
    def __len__(self):
        return len(self.index_map)
    
    def __getitem__(self, idx):
        vi, si = self.index_map[idx]
        vol, msk = self._load(vi)
        img = vol[:, :, si].copy().astype(np.float32)
        gt = msk[:, :, si].copy().astype(np.int64)
        
        if self.transform is not None:
            t = self.transform(image=img, mask=gt)
            img, gt = t['image'], t['mask']
        
        img = torch.from_numpy(img).unsqueeze(0).float()
        if self.in_channels == 3:
            img = img.repeat(3, 1, 1)
        return img, torch.from_numpy(gt).long() if isinstance(gt, np.ndarray) else gt.long()

In [ ]:
# Build datasets
base_ds = SliceDataset(train_dir, in_channels=cfg['in_channels'], augment=False)

num_vols = len(base_ds.vol_paths)
vol_idx = list(range(num_vols))
np.random.seed(42)
np.random.shuffle(vol_idx)
split = int(num_vols * 0.8)
train_vols, val_vols = set(vol_idx[:split]), set(vol_idx[split:])

train_idx = [i for i, (v, s) in enumerate(base_ds.index_map) if v in train_vols]
val_idx = [i for i, (v, s) in enumerate(base_ds.index_map) if v in val_vols]

if TRAIN_CFG['augment']:
    train_ds = SliceDataset(train_dir, in_channels=cfg['in_channels'], augment=True)
    train_ds.index_map = [base_ds.index_map[i] for i in train_idx]
else:
    train_ds = Subset(base_ds, train_idx)

val_ds = Subset(base_ds, val_idx)

train_loader = DataLoader(train_ds, TRAIN_CFG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, TRAIN_CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} slices | Val: {len(val_ds)} slices')

## 4. Model Architecture

In [ ]:
# ============================================================
# Building blocks — SpectralBlock, PseudoMambaBlock, SpecMambaBlock
# ============================================================


class SpectralBlock(nn.Module):
    """Channel mixing in Fourier domain using real FFT."""
    def __init__(self, dim):
        super().__init__()
        self.conv_real = nn.Conv2d(dim, dim, 1, bias=False)
        self.conv_imag = nn.Conv2d(dim, dim, 1, bias=False)
        self.norm = nn.GroupNorm(min(8, dim), dim)
        self.act = nn.GELU()

    def forward(self, x):
        B, C, H, W = x.shape
        with torch.amp.autocast('cuda', enabled=False):
            x_f = x.float()
            x_freq = torch.fft.rfft2(x_f, norm='ortho')
            out_real = self.conv_real(x_freq.real)
            out_imag = self.conv_imag(x_freq.imag)
            x_freq_mixed = torch.complex(out_real, out_imag)
            x_out = torch.fft.irfft2(x_freq_mixed, s=(H, W), norm='ortho')
        return self.act(self.norm(x_out))


class PseudoMambaBlock(nn.Module):
    """SSM-style block using causal depthwise Conv1d + gating (pure PyTorch)."""
    def __init__(self, dim, kernel_size=3):
        super().__init__()
        self.dim = dim
        self.linear_in = nn.Linear(dim, dim, bias=False)
        self.linear_gate = nn.Linear(dim, dim, bias=False)
        self.dw_conv = nn.Conv1d(
            dim, dim,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
            groups=dim,
            bias=False,
        )
        self.linear_out = nn.Linear(dim, dim, bias=False)
        self.norm = nn.GroupNorm(min(8, dim), dim)
        self.act = nn.GELU()

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        seq = x.view(B, C, N).transpose(1, 2)
        h = self.act(self.linear_in(seq))
        g = torch.sigmoid(self.linear_gate(seq))
        h_t = h.transpose(1, 2)
        h_t = self.dw_conv(h_t)[..., :N]
        h = h_t.transpose(1, 2)
        h = h * g
        out = self.linear_out(h)
        x_out = out.transpose(1, 2).view(B, C, H, W)
        return self.act(self.norm(x_out))


class SpecMambaBlock(nn.Module):
    """Dual-stream block: spectral (FFT) + pseudo-Mamba, fused with residual."""
    def __init__(self, dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, dim), dim)
        self.norm2 = nn.GroupNorm(min(8, dim), dim)
        self.spectral = SpectralBlock(dim)
        self.mamba = PseudoMambaBlock(dim)
        self.fuse = nn.Conv2d(dim * 2, dim, 1, bias=False)
        self.norm_out = nn.GroupNorm(min(8, dim), dim)
        self.act = nn.GELU()

    def forward(self, x):
        x_spec = self.spectral(self.norm1(x))
        x_mamba = self.mamba(self.norm2(x))
        x_fused = self.fuse(torch.cat([x_spec, x_mamba], dim=1))
        return x + self.act(self.norm_out(x_fused))

In [ ]:
# ============================================================
# SpecMambaNet — U-Net Encoder-Decoder with SpecMambaBlock
# ============================================================


class SpecMambaNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=4, base_channels=48,
                 img_size=224, deep_supervision=False):
        super().__init__()
        self.deep_supervision = deep_supervision
        C = base_channels

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, C, 3, 1, 1, bias=False),
            nn.GroupNorm(min(8, C), C),
            nn.GELU(),
        )

        # Encoder
        self.enc1 = SpecMambaBlock(C)
        self.down1 = nn.Sequential(
            nn.Conv2d(C, C * 2, 3, stride=2, padding=1, bias=False),
            nn.GroupNorm(min(8, C * 2), C * 2), nn.GELU(),
        )
        self.enc2 = SpecMambaBlock(C * 2)
        self.down2 = nn.Sequential(
            nn.Conv2d(C * 2, C * 4, 3, stride=2, padding=1, bias=False),
            nn.GroupNorm(min(8, C * 4), C * 4), nn.GELU(),
        )
        self.enc3 = SpecMambaBlock(C * 4)
        self.down3 = nn.Sequential(
            nn.Conv2d(C * 4, C * 8, 3, stride=2, padding=1, bias=False),
            nn.GroupNorm(min(8, C * 8), C * 8), nn.GELU(),
        )
        self.bottleneck = SpecMambaBlock(C * 8)

        # Decoder
        self.up3_fuse = nn.Conv2d(C * 8 + C * 4, C * 4, 1)
        self.dec3 = SpecMambaBlock(C * 4)

        self.up2_fuse = nn.Conv2d(C * 4 + C * 2, C * 2, 1)
        self.dec2 = SpecMambaBlock(C * 2)

        self.up1_fuse = nn.Conv2d(C * 2 + C, C, 1)
        self.dec1 = SpecMambaBlock(C)

        # Segmentation head
        self.seg_head = nn.Conv2d(C, num_classes, 1)

        # Deep supervision aux heads
        if deep_supervision:
            self.aux_head_3 = nn.Conv2d(C * 4, num_classes, 1)
            self.aux_head_2 = nn.Conv2d(C * 2, num_classes, 1)
            self.aux_head_1 = nn.Conv2d(C, num_classes, 1)

    def forward(self, x):
        target = x.shape[2:]

        x = self.stem(x)

        feat1 = self.enc1(x)
        feat2 = self.enc2(self.down1(feat1))
        feat3 = self.enc3(self.down2(feat2))
        feat4 = self.bottleneck(self.down3(feat3))

        up3 = F.interpolate(feat4, scale_factor=2, mode='bilinear', align_corners=True)
        dec3 = self.dec3(self.up3_fuse(torch.cat([up3, feat3], dim=1)))

        up2 = F.interpolate(dec3, scale_factor=2, mode='bilinear', align_corners=True)
        dec2 = self.dec2(self.up2_fuse(torch.cat([up2, feat2], dim=1)))

        up1 = F.interpolate(dec2, scale_factor=2, mode='bilinear', align_corners=True)
        dec1 = self.dec1(self.up1_fuse(torch.cat([up1, feat1], dim=1)))

        logits = F.interpolate(self.seg_head(dec1), target, mode='bilinear', align_corners=True)

        if self.deep_supervision and self.training:
            aux3 = F.interpolate(self.aux_head_3(dec3), target, mode='bilinear', align_corners=True)
            aux2 = F.interpolate(self.aux_head_2(dec2), target, mode='bilinear', align_corners=True)
            aux1 = F.interpolate(self.aux_head_1(dec1), target, mode='bilinear', align_corners=True)
            return {'output': logits, 'aux_outputs': [aux3, aux2, aux1]}

        return {'output': logits}

In [ ]:
# Build model
model = SpecMambaNet(
    in_channels=cfg['in_channels'],
    num_classes=cfg['num_classes'],
    base_channels=TRAIN_CFG['base_channels'],
    img_size=cfg['img_size'],
    deep_supervision=TRAIN_CFG['deep_supervision'],
).to(DEVICE)

params = sum(p.numel() for p in model.parameters())
print(f'Model: SpecMambaNet-C{TRAIN_CFG["base_channels"]} | Params: {params:,}')

In [ ]:
# Smoke test
model.eval()
dummy = torch.randn(2, cfg['in_channels'], cfg['img_size'], cfg['img_size']).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
assert out['output'].shape == (2, cfg['num_classes'], cfg['img_size'], cfg['img_size']), \
    f"Shape mismatch: {out['output'].shape}"
print("✓ Forward pass OK:", out['output'].shape)

model.train()
out_train = model(dummy)
if TRAIN_CFG['deep_supervision']:
    assert 'aux_outputs' in out_train, "Missing aux_outputs!"
    for i, a in enumerate(out_train['aux_outputs']):
        assert a.shape == out['output'].shape, f"Aux {i} shape: {a.shape}"
    print(f"✓ Deep supervision OK: {len(out_train['aux_outputs'])} aux heads")

print(f"✓ Params: {params:,}")

## 5. Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5, class_weights=None):
        super().__init__()
        self.smooth = smooth
        self.class_weights = class_weights

    def forward(self, pred, target):
        pred = torch.softmax(pred, 1)
        nc = pred.shape[1]
        if target.ndim == 3:
            target = F.one_hot(target.long(), nc).permute(0,3,1,2).float()
        p = pred.view(pred.shape[0], nc, -1)
        t = target.view(target.shape[0], nc, -1)
        inter = (p * t).sum(2)
        union = p.sum(2) + t.sum(2)
        dice = (2 * inter + self.smooth) / (union + self.smooth)
        if self.class_weights is not None:
            w = self.class_weights.to(pred.device)
            return ((1 - dice) * w.view(1, -1)).sum(1).mean() / w.sum()
        return (1 - dice).mean()


class BoundaryLoss(nn.Module):
    """Distance-based boundary loss (Kervadec 2019)."""
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        pred_probs = torch.softmax(pred, 1)
        nc = pred_probs.shape[1]
        total = 0
        for c in range(1, nc):
            tc = (target == c).float()
            pc = pred_probs[:, c]
            dist_maps = []
            for b in range(tc.shape[0]):
                m = tc[b].cpu().numpy()
                d = np.zeros_like(m, dtype=np.float32) if m.sum() == 0 else distance_transform_edt(1-m) + distance_transform_edt(m)
                dist_maps.append(d)
            dm = torch.from_numpy(np.stack(dist_maps)).float().to(pred.device)
            total += ((pc - tc).abs() * dm).mean()
        return total / max(nc - 1, 1)


class CombinedLoss(nn.Module):
    def __init__(self, num_classes, ce_w=1.0, dice_w=1.0, bnd_w=0.5,
                 warmup=10, class_weights=None):
        super().__init__()
        self.ce_w, self.dice_w, self.bnd_w = ce_w, dice_w, bnd_w
        self.warmup = warmup
        self.epoch = 0
        self._cw = class_weights
        self.dice = DiceLoss()
        self.bnd = BoundaryLoss()

    def forward(self, pred, target):
        dev = pred.device
        cw = torch.tensor(self._cw, dtype=torch.float32, device=dev) if self._cw else None
        
        ce_pp = F.cross_entropy(pred, target.long(), reduction='none')
        ce = (ce_pp * cw[target.long()]).mean() if cw is not None else ce_pp.mean()
        
        self.dice.class_weights = cw
        dice = self.dice(pred, target)
        
        loss = self.ce_w * ce + self.dice_w * dice
        ld = {'ce': ce.item(), 'dice': dice.item()}
        
        if self.epoch >= self.warmup // 2:
            wf = min(1.0, (self.epoch - self.warmup // 2) / self.warmup)
            bl = self.bnd(pred, target)
            loss += self.bnd_w * wf * bl
            ld['bnd'] = bl.item()
        
        ld['total'] = loss.item()
        return loss, ld

## 6. Evaluation (Spacing-aware HD95 in mm)

In [ ]:
def load_volume_meta(data_dir):
    """Load volume metadata with spacing info."""
    mp = os.path.join(data_dir, 'metadata.json')
    if not os.path.exists(mp):
        return {}
    with open(mp) as f:
        vi = json.load(f).get('volume_info', {})
    meta = {}
    for idx, vname in enumerate(sorted(vi.keys())):
        if 'effective_spacing' in vi[vname]:
            meta[idx] = vi[vname]
    return meta


def evaluate_3d(model, dataset, device, num_classes, volume_meta=None):
    """3D volumetric evaluation with spacing-aware HD95 (mm)."""
    model.eval()
    vol_preds, vol_targets = defaultdict(list), defaultdict(list)
    
    with torch.no_grad():
        for i in range(len(dataset)):
            vi, si = dataset.dataset.index_map[dataset.indices[i]]
            img, tgt = dataset[i]
            pred = model(img.unsqueeze(0).to(device))['output'].argmax(1).squeeze(0).cpu().numpy()
            vol_preds[vi].append((si, pred))
            vol_targets[vi].append((si, tgt.numpy()))
    
    metrics = {c: {'dice': [], 'hd95': [], 'prec': [], 'rec': []} for c in range(1, num_classes)}
    
    for vi in vol_preds:
        p3d = np.stack([p[1] for p in sorted(vol_preds[vi])], 0)
        t3d = np.stack([t[1] for t in sorted(vol_targets[vi])], 0)
        sp = volume_meta[vi]['effective_spacing'] if volume_meta and vi in volume_meta else None
        
        for c in range(1, num_classes):
            pc, tc = (p3d == c), (t3d == c)
            if not tc.any():
                if pc.any():
                    metrics[c]['dice'].append(0.0); metrics[c]['hd95'].append(100.0)
                    metrics[c]['prec'].append(0.0); metrics[c]['rec'].append(0.0)
                continue
            
            tp = (pc & tc).sum(); fp = (pc & ~tc).sum(); fn = (~pc & tc).sum()
            metrics[c]['dice'].append((2*tp) / (2*tp + fp + fn + 1e-6))
            metrics[c]['prec'].append(tp / (tp + fp + 1e-6))
            metrics[c]['rec'].append(tp / (tp + fn + 1e-6))
            
            if pc.any() and tc.any():
                pb = pc ^ binary_erosion(pc); tb = tc ^ binary_erosion(tc)
                if not pb.any(): pb = pc
                if not tb.any(): tb = tc
                d1 = distance_transform_edt(~tc, sampling=sp)[pb]
                d2 = distance_transform_edt(~pc, sampling=sp)[tb]
                metrics[c]['hd95'].append(np.percentile(np.concatenate([d1, d2]), 95))
            else:
                metrics[c]['hd95'].append(100.0)
    
    sm = lambda lst: np.mean(lst) if lst else 0.0
    fg = range(1, num_classes)
    return {
        'dice': np.mean([sm(metrics[c]['dice']) for c in fg]),
        'hd95': np.mean([sm(metrics[c]['hd95']) for c in fg]),
        'prec': np.mean([sm(metrics[c]['prec']) for c in fg]),
        'rec':  np.mean([sm(metrics[c]['rec'])  for c in fg]),
        'per_class': {c: {k: sm(v) for k, v in metrics[c].items()} for c in fg},
    }


vol_meta = load_volume_meta(train_dir)
print(f'Spacing loaded: {len(vol_meta)} volumes' if vol_meta else 'No spacing (HD95 in voxels)')

## 7. Training Loop

In [ ]:
# Class weights per dataset
CLASS_WEIGHTS = {
    'acdc': [0.1, 1.5, 1.5, 1.0],    # BG, RV, MYO, LV
    'imagecas': [0.2, 1.0],            # BG, Vessel
}

criterion = CombinedLoss(
    num_classes=cfg['num_classes'],
    class_weights=CLASS_WEIGHTS[DATASET],
    warmup=TRAIN_CFG['warmup_epochs'],
)

optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_CFG['lr'], weight_decay=TRAIN_CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_CFG['epochs'], eta_min=1e-7)
scaler = torch.amp.GradScaler('cuda') if TRAIN_CFG['use_amp'] else None

DS_WEIGHTS = [0.4, 0.3, 0.2, 0.1]
ALPHA_HD95 = 0.7

best = {'dice': 0, 'hd95': float('inf'), 'balanced': float('-inf')}
no_improve = 0
history = defaultdict(list)

save_dir = '/kaggle/working/weights'
os.makedirs(save_dir, exist_ok=True)
exp_name = f'{DATASET}_specmamba_c{TRAIN_CFG["base_channels"]}'

print(f'\nTraining: {TRAIN_CFG["epochs"]} epochs | BS={TRAIN_CFG["batch_size"]} | LR={TRAIN_CFG["lr"]}')
print(f'Balanced Score = Dice - {ALPHA_HD95}*HD95\n')

In [ ]:
for epoch in range(TRAIN_CFG['epochs']):
    criterion.epoch = epoch
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f'E{epoch+1:03d}', leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        
        if TRAIN_CFG['use_amp'] and scaler:
            with torch.amp.autocast('cuda'):
                out = model(imgs)
                loss, ld = criterion(out['output'], masks)
                if TRAIN_CFG['deep_supervision'] and 'aux_outputs' in out:
                    for i, aux in enumerate(out['aux_outputs']):
                        al, _ = criterion(aux, masks)
                        loss = loss + DS_WEIGHTS[i] * al
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(imgs)
            loss, ld = criterion(out['output'], masks)
            if TRAIN_CFG['deep_supervision'] and 'aux_outputs' in out:
                for i, aux in enumerate(out['aux_outputs']):
                    al, _ = criterion(aux, masks)
                    loss = loss + DS_WEIGHTS[i] * al
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    
    # Evaluate
    m = evaluate_3d(model, val_ds, DEVICE, cfg['num_classes'], vol_meta)
    dice, hd95 = m['dice'], m['hd95']
    penalty = 10.0 if hd95 > 100 or np.isnan(hd95) else hd95
    balanced = dice - ALPHA_HD95 * penalty
    
    # Log
    history['loss'].append(avg_loss); history['dice'].append(dice)
    history['hd95'].append(hd95); history['balanced'].append(balanced)
    
    lr = optimizer.param_groups[0]['lr']
    print(f'E{epoch+1:03d} | Loss: {avg_loss:.4f} | Dice: {dice:.4f} | HD95: {hd95:.2f}mm | Bal: {balanced:.4f} | LR: {lr:.6f}')
    for c in range(1, cfg['num_classes']):
        pc = m['per_class'][c]
        print(f'  {cfg["class_names"][c]:>6}: Dice={pc["dice"]:.4f} HD95={pc["hd95"]:.2f}mm Prec={pc["prec"]:.4f} Rec={pc["rec"]:.4f}')
    
    # Save best
    saved = False
    if dice > best['dice']:
        best['dice'] = dice
        torch.save(model.state_dict(), f'{save_dir}/{exp_name}_best_dice.pt')
        print(f'  ★ Best Dice: {dice:.4f}'); saved = True
    if hd95 < best['hd95']:
        best['hd95'] = hd95
        torch.save(model.state_dict(), f'{save_dir}/{exp_name}_best_hd95.pt')
        print(f'  ★ Best HD95: {hd95:.2f}mm'); saved = True
    if balanced > best['balanced']:
        best['balanced'] = balanced
        torch.save(model.state_dict(), f'{save_dir}/{exp_name}_best_balanced.pt')
        print(f'  ★ Best Balanced: {balanced:.4f}'); saved = True
    
    no_improve = 0 if saved else no_improve + 1
    if no_improve >= TRAIN_CFG['early_stop']:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

print(f'\nDone! Best Dice: {best["dice"]:.4f} | HD95: {best["hd95"]:.2f}mm | Balanced: {best["balanced"]:.4f}')

## 8. Final Evaluation on Test Set

In [ ]:
# Load best model and evaluate on test set
test_dir = os.path.join(cfg['prep_dir'], 'testing')

if os.path.isdir(test_dir) and glob.glob(os.path.join(test_dir, 'volumes', '*.npy')):
    ckpt_path = f'{save_dir}/{exp_name}_best_balanced.pt'
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    print(f'Loaded: {ckpt_path}')
    
    test_base = SliceDataset(test_dir, in_channels=cfg['in_channels'], augment=False)
    test_ds = Subset(test_base, list(range(len(test_base))))
    test_meta = load_volume_meta(test_dir)
    
    m = evaluate_3d(model, test_ds, DEVICE, cfg['num_classes'], test_meta)
    
    print(f'\n{"="*50}')
    print(f'TEST RESULTS ({DATASET.upper()})')
    print(f'{"="*50}')
    print(f'Mean Dice: {m["dice"]:.4f}')
    print(f'Mean HD95: {m["hd95"]:.2f} mm')
    print(f'Mean Prec: {m["prec"]:.4f}')
    print(f'Mean Rec:  {m["rec"]:.4f}')
    print(f'\nPer-Class:')
    for c in range(1, cfg['num_classes']):
        pc = m['per_class'][c]
        print(f'  {cfg["class_names"][c]:>6}: Dice={pc["dice"]:.4f}  HD95={pc["hd95"]:.2f}mm  Prec={pc["prec"]:.4f}  Rec={pc["rec"]:.4f}')
else:
    print('No test set found. Skipping final evaluation.')

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True)

axes[1].plot(history['dice'], label='Dice')
axes[1].set_title('Validation Dice'); axes[1].set_xlabel('Epoch'); axes[1].grid(True); axes[1].legend()

axes[2].plot(history['hd95'], label='HD95 (mm)', color='red')
axes[2].set_title('Validation HD95 (mm)'); axes[2].set_xlabel('Epoch'); axes[2].grid(True); axes[2].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Save history
with open(f'{save_dir}/{exp_name}_history.json', 'w') as f:
    json.dump(dict(history), f, indent=2)
print('History saved.')